# 実験音韻論の方法 ── 音韻理論をデータで検証する

### 第6章 分析の自動化 ── 分析コード集


ここに掲載されているコードは、
『実験音韻論の方法 ── 音韻理論をデータで検証する』(音声学ライブラリ 第4巻)の
分析コード集の一部である。MIT Licenseで公開しており、自由に複製・改変・
再配布してよい(詳細はサポートリポジトリのLICENSEを参照)。

このファイルを開いた時点では閲覧用として開くが、**ファイル → ドライブに
コピーを保存**を行えば、あなた自身のGoogle Driveに保存されたあなただけの
ファイルになる。自由に変更・実行・保存してよい。

**本文で報告されている数値・図は、すべてこのノートブックのコードを実際に
実行して得られたものである。** 架空データ(シミュレーション)を使っている
箇所は、各コードセルの直前のセルにその旨を明記してある。実データを模した
説明にすぎない部分と、実際の統計的性質を確認するための実行例とを、
本書は厳格に区別しており、このノートブックもその区別をそのまま保持している。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で
言語を **R** に切り替えること。既定のPythonランタイムのままだと
以降のセルがすべてエラーになる。

**生成される図・CSVの保存場所について。** コード中に出てくる
`data/processed/`や`assets/figures/`は、あなたのGoogle Driveの中の
フォルダではない。Colabの仮想マシン上に一時的に作られるだけの作業用
フォルダであり、**セッションが切断される(一定時間の無操作、ランタイムの
再起動など)と中身ごと消える。** 保存したい図やCSVがあれば、Colab画面
左側のファイル一覧からその都度手元のパソコンにダウンロードするか、
コードを`/content/drive/...`のようなGoogle Drive上のパスに書き出すよう
自分で書き換える必要がある。


## 共通セットアップ

以降の各セルは、本書の`scripts/R/`に収録されている実際のコードと一字一句同じである(差し替えたのはこのセットアップだけ)。まずこのセルを実行してから、上から順に進めること。

In [ ]:
# ---- 図の日本語文字化け対策 ----
# Colabの既定Rランタイムのフォントには日本語グリフが無く、図中の日本語
# (タイトル・軸ラベルなど)が文字化け(豆腐記号)して表示される。showtextで
# Google Fontsの日本語フォントを取得し、"notosans"という名前で登録することで
# 回避する("sans"などの予約名はshowtextから上書きできないため専用名で登録する)。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")
showtext_auto()
showtext_opts(dpi = 100)

# ---- 共通セットアップ(scripts/R/00_setup.RのColab版) ----
# 本書のすべての解析スクリプトが冒頭で読み込む初期化処理を、Colab向けに
# 移植したもの。以降の章のコードは、本書のscripts/R/chNN_*.Rと一字一句同じ
# ものを、このセットアップの後に続けて実行する形になっている。

pkgs <- c("tidyverse", "lme4", "lmerTest", "mgcv", "itsadug",
          "phonTools", "randomForest", "patchwork", "here")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lme4)
library(lmerTest)
library(here)

set.seed(20260101)   # 乱数を使う処理(ブートストラップ、MCMC)の再現のため(第9章参照)

theme_labphon <- function(base_size = 11) {
  theme_minimal(base_size = base_size, base_family = "notosans") +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(linewidth = 0.3, colour = "grey88"),
      axis.title       = element_text(size = rel(0.95)),
      strip.text       = element_text(face = "bold", size = rel(0.9)),
      legend.position  = "bottom",
      plot.title        = element_text(face = "bold", size = rel(1.05)),
      plot.caption      = element_text(size = rel(0.8), colour = "grey35")
    )
}
theme_set(theme_labphon())

# 本書のsave_fig()はPNG/PDFをassets/figures/に書き出すが、このノートブックでは
# 画面にそのまま表示すれば十分なので、print()する版に差し替える。
# (関数のシグネチャは本書と同じなので、以降のスクリプトは書き換えなしで動く。)
save_fig <- function(plot, name, width = 5.2, height = 3.6, chapter = NULL) {
  print(plot)
  invisible(NULL)
}

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
# 一部のスクリプトはsave_fig()を経由せず直接ggsave()/png()でassets/figures/に
# 書き出す(このディレクトリ自体はColab上のこのセッション内でのみ存在する
# 一時的なものであり、書籍リポジトリの実ファイルには一切影響しない)。
for (chn in sprintf("ch%02d", 2:16)) {
  dir.create(here::here("assets", "figures", chn), recursive = TRUE, showWarnings = FALSE)
}

cat("セットアップ完了。実行環境:\n")
print(sessionInfo())


## `ch06_pipeline.R`

対象章: 第6章「分析の自動化」

「Praat Scriptの基礎」節のパイプライン全体図。生の音声ファイルから
測定値テーブルに至るまでの処理の流れを、本章が扱う技術要素
(Praat Script、Parselmouth、強制アラインメント、品質管理)ごとに
図示する概念図。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

nodes <- tibble(
  x     = 1,
  y     = 6:1,
  label = c(
    "音声ファイル群(.wav)\n+ 書き起こしテキスト",
    "強制アラインメント(MFA)\n→ 音素・単語ごとのTextGrid",
    "Praat Script / Parselmouth\nバッチ測定(F0・フォルマント・持続時間等)",
    "自動測定の品質管理\n(異常値検出、信頼度の低い区間の除外)",
    "処理の来歴の記録\n(バージョン・パラメータ・実行日時)",
    "測定値テーブル(CSV)\n→ 第9章以降の統計解析へ"
  )
)

p <- ggplot(nodes) +
  geom_rect(aes(xmin = 0, xmax = 10, ymin = y - 0.35, ymax = y + 0.35),
            fill = "#34518A", alpha = 0.12, colour = "#34518A") +
  geom_text(aes(x = 5, y = y, label = label), size = 3, colour = "#1B2333", lineheight = 0.9) +
  geom_segment(data = nodes |> filter(y > 1),
               aes(x = 5, xend = 5, y = y - 0.38, yend = y - 0.65),
               arrow = arrow(length = unit(0.18, "cm"), type = "closed"), colour = "#A8681F") +
  scale_x_continuous(limits = c(0, 10)) +
  scale_y_continuous(limits = c(0.5, 6.5)) +
  theme_void() +
  labs(caption = "概念図。scripts/R/ch06_pipeline.R より生成。") +
  theme(plot.caption = element_text(size = rel(0.8), colour = "grey35"))

save_fig(p, "fig06-01-pipeline-diagram", chapter = "ch06", width = 6, height = 8)
cat("完了。図はassets/figures/ch06/に保存した。\n")


## `ch06_mfa_bias_propagation.R`

対象章: 第6章「分析の自動化」

「アライメント精度の限界と検証」節の作例。MFAの境界誤差が音素クラスに
よって系統的に異なる場合(閉鎖音は誤差が小さくランダム、摩擦音は
誤差が大きく系統的に短く出る)、その誤差が下流の統計的結論(2つの
音素クラス間の持続時間の差)にどう伝播するかを、実際にシミュレーション
で確認する。真の持続時間はどちらのクラスも同一(100ms)に設定してあり、
観測される差はすべてMFAの境界誤差に由来する。架空のデータであり、
実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n_tokens <- 200
true_duration <- 100   # ミリ秒、両クラスで同一の真の持続時間

# 閉鎖音: MFAの境界誤差はランダムでほぼ0を中心とする
stop_measured <- true_duration + rnorm(n_tokens, 0, 5)

# 摩擦音: 遷移がなだらかなため、MFAは境界を早めに(短く)検出しがち
fricative_measured <- true_duration - 20 + rnorm(n_tokens, 0, 5)

cat("真の持続時間: 両クラスとも", true_duration, "ms(同一)\n\n")
cat("閉鎖音(MFA境界誤差はランダム): 測定平均 =",
    round(mean(stop_measured), 1), "ms\n")
cat("摩擦音(MFA境界誤差が系統的に-20ms): 測定平均 =",
    round(mean(fricative_measured), 1), "ms\n")

test <- t.test(fricative_measured, stop_measured)
cat("\n2群の測定値の差の検定: 差 =", round(diff(test$estimate) * -1, 1),
    "ms, p =", format.pval(test$p.value, digits = 3), "\n")
cat("(真の差は0msだが、MFAの境界誤差だけで統計的に有意な差が生じている)\n")


## `ch06_dict_coverage.R`

対象章: 第6章「分析の自動化」

「強制アラインメント(MFA)の原理と手順」節の作例。発音辞書が話し言葉の
縮約形をどれだけ収録しているかによって、実際に整合できるトークンの
割合がどう変わるかを、架空の語彙頻度分布で確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(tibble); library(dplyr)

set.seed(20260106)
n_tokens <- 2000

# 上位20語(高頻度語)は縮約形が生じやすく、残りの語彙は縮約が稀とする
word_rank <- sample(1:200, n_tokens, replace = TRUE,
                     prob = c(rep(5, 20), rep(1, 180)))
is_high_freq <- word_rank <= 20

# 縮約が生じる確率: 高頻度語で40%、それ以外で5%
contracted <- ifelse(is_high_freq, rbinom(n_tokens, 1, 0.40),
                                    rbinom(n_tokens, 1, 0.05))

dict_covers_contraction <- FALSE   # 標準的な発音辞書は縮約形を収録していないと仮定
aligned_ok <- !(contracted == 1 & !dict_covers_contraction)

cat("総トークン数:", n_tokens, "\n")
cat("縮約形が生じたトークン数:", sum(contracted), "(",
    round(mean(contracted) * 100, 1), "%)\n")
cat("縮約形のうち高頻度語(上位20語)由来の割合:",
    round(mean(word_rank[contracted == 1] <= 20) * 100, 1), "%\n")
cat("辞書に縮約形がないために整合できなかったトークン数:",
    sum(!aligned_ok), "(全体の", round(mean(!aligned_ok) * 100, 1), "%)\n")


## `ch06_outlier_detection.R`

対象章: 第6章「分析の自動化」

「自動測定の品質管理」節の作例。平均・標準偏差に基づく外れ値検出と、
中央値・MAD(中央値絶対偏差)に基づく外れ値検出が、外れ値そのものが
多く紛れ込むデータに対してどう異なる挙動を示すかを、架空のデータで
確認する。実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260906)

n_normal  <- 260
n_extreme <- 60   # 測定失敗由来の極端な値(例: ピッチのオクターブエラー)。混入率を高めに設定

normal_vals  <- rnorm(n_normal, mean = 150, sd = 15)     # 正常なF0測定値(Hz)
extreme_vals <- rnorm(n_extreme, mean = 290, sd = 15)    # オクターブエラー由来の異常値(約2倍)

d <- tibble(f0 = c(normal_vals, extreme_vals),
            true_status = c(rep("正常", n_normal), rep("測定失敗", n_extreme)))

# --- 方法1: 平均±3SD -----------------------------------------------------
mean_sd_flag <- abs(d$f0 - mean(d$f0)) > 3 * sd(d$f0)

# --- 方法2: 中央値±3MAD(MADは正規分布に合わせて1.4826倍で標準化) --------
med <- median(d$f0)
mad_val <- mad(d$f0)   # Rのmad()はデフォルトで1.4826倍済み
mad_flag <- abs(d$f0 - med) > 3 * mad_val

d <- d |> mutate(mean_sd_flag = mean_sd_flag, mad_flag = mad_flag)

# 検出率(真の異常値のうち、実際に検出された割合)と誤検出率を集計
summarize_method <- function(flag) {
  tibble(
    検出された異常値数 = sum(flag & d$true_status == "測定失敗"),
    異常値の検出率     = sum(flag & d$true_status == "測定失敗") / n_extreme,
    正常値の誤検出数   = sum(flag & d$true_status == "正常"),
    誤検出率           = sum(flag & d$true_status == "正常") / n_normal
  )
}

cat("=== 平均±3SD法 ===\n")
print(summarize_method(d$mean_sd_flag))
cat("平均:", round(mean(d$f0), 1), " SD:", round(sd(d$f0), 1), "\n\n")

cat("=== 中央値±3MAD法 ===\n")
print(summarize_method(d$mad_flag))
cat("中央値:", round(med, 1), " MAD:", round(mad_val, 1), "\n")


## `ch06_checkpoint_resume.R`

対象章: 第6章「分析の自動化」

「処理の来歴とチェックポイント」節の作例。すでに処理済みのファイルを
記録しておき、パイプラインが中断してから再実行する際に、処理済みの
ファイルをスキップして未処理のファイルだけを処理する仕組みを、
架空のファイル一覧に対して実際に動かして確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(tibble); library(dplyr)

all_files <- sprintf("spk%02d_item%02d.wav", rep(1:10, each = 5), rep(1:5, 10))
cat("処理対象の総ファイル数:", length(all_files), "\n")

# 1回目の実行: 途中(37件目)で中断したと仮定する
processed_log <- all_files[1:37]
cat("1回目の実行で処理済み(中断まで):", length(processed_log), "件\n")

# 2回目の実行: ログに記録されたファイルをスキップし、残りだけを処理する
remaining <- setdiff(all_files, processed_log)
cat("2回目の実行でスキップされるファイル数:", length(all_files) - length(remaining), "\n")
cat("2回目の実行で新たに処理されるファイル数:", length(remaining), "\n")
cat("先頭3件:", paste(head(remaining, 3), collapse = ", "), "\n")
